# Generate Movie Reviews using LLM:A comprehensive Demo

In this notebook, we use the 3.8B parameter [Phi-3 model](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct) to write move reviews.

We will also take a closer look at the inputs and outputs of the tokenizer and the LLM, to understand what is happening under the hood when we call a LLM text generation API.

We also look at the effect of `temperature`, a common parameter used to control LLM text generation randomness. Higher temperature gives more random/creative outputs, while lower, near zero gives less varied outputs. There is also a "greedy" text generation strategy: under this strategy, the vocab token with the highest next token probability will always be selected, making the generation deterministic.



## Step 1: Import Libraries

In [ ]:
import torch
import transformers

transformers.utils.logging.set_verbosity_error()
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Step 2:Download a pretrained LLM from Huggingface

In [ ]:
#@title Download phi-3 model and its tokenizer from hugging face

from transformers import AutoModelForCausalLM, AutoTokenizer

#Load <pre-trained>Phi3 Model
phi3_model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3-mini-4k-instruct",
                                             torch_dtype="auto",
                                             trust_remote_code=False
                                             )
phi3_tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Step 3:A Util Function to generate LLM response given a prompt

- ### Tokenization(Breaks Sentence to Words and to input tokens).
- ### Send this tokens to a model(Input prompt sent as tokens to the Model)
- ### Generate method to get the decoded output:Parameters to `generate`:
  - `temperature`: Controls Randomness of Output.
  Higher the temperature more random the output, leading to a more Creative Model.
  Lower temperature results in a more determistic model and very coherent output.
  - `max_new_tokens`: Maximum New tokens to generate (ignoring the tokens in prompt).


In [ ]:
# Our text generation "API".
def llm_generate_text(model, tokenizer, prompt, temperature=0.8, max_new_tokens=256):

  #Tokenize the Natural Language text and return `pytorch` format tensors
  tokenized_inputs = tokenizer(prompt, return_tensors="pt")
  #Send tokenized ip to `device`(GPU/CPU)
  tokenized_inputs = {k: v.to(device) for k, v in tokenized_inputs.items()}
  #Send Model to `device`(GPU/CPU)
  model = model.to(device)
  ##Temperature to control randomness of model
  predicted_token_ids = model.generate(
      **tokenized_inputs,#Input Ids(Tokenized Prompt)
      do_sample=True,
      temperature=temperature,#Controls randomness of output
      max_new_tokens=max_new_tokens,#Maximum Tokens Model can generate
  )[0]

  print(tokenizer.decode(predicted_token_ids))

## Step 4: Generate movie review using phi3 model

* We ask the model to write a movie review starting with "I".
* The phi3 model expects chat formatted input. We use `tokenizer.apply_chat_template` to easily do this.

It's easy to see that phi3's movie reviews are vastly better than our previous bi-gram and 4-gram models. Importantly, we did not need to train the phi3 model either: the model has gone through sufficient prior training to satisfy our request.

In [ ]:
prompt = 'Write a movie review. The movie should be an actual movie. The review should start with the word "I".'

#Applying Chat Template
messages = [{"role": "user", "content": prompt}]
phi3_prompt = phi3_tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
print(phi3_prompt)

llm_generate_text(phi3_model, phi3_tokenizer, phi3_prompt)

<|user|>
Write a movie review. The movie should be an actual movie. The review should start with the word "I".<|end|>
<|assistant|>

<|user|> Write a movie review. The movie should be an actual movie. The review should start with the word "I".<|end|><|assistant|> I recently had the pleasure of watching "The Shawshank Redemption", a cinematic masterpiece that is both moving and uplifting. Directed by Frank Darabont, the film tells the story of Andy Dufresne, a banker who is sentenced to life at Shawshank State Penitentiary for the murder of his wife and her lover, a crime he didn't commit.

From the very beginning, the movie captures the audience's attention with its compelling soundtrack and stunning visuals. The portrayal of the prison is vivid and realistic, showcasing the harsh conditions faced by its inmates. I was immediately pulled into the lives of the characters, particularly the relationship between Andy and Red, played brilliantly by Tim Robbins and Morgan Freeman, respective

### Now let's look at the `llm_generate_text` function line by line.
```
def llm_generate_text(model, tokenizer, prompt, temperature=0.8, max_new_tokens=256):
  tokenized_inputs = tokenizer(prompt, return_tensors="pt")
  tokenized_inputs = {k: v.to(device) for k, v in tokenized_inputs.items()}

  model = model.to(device)

  predicted_token_ids = model.generate(
      **tokenized_inputs,
      do_sample=True,
      temperature=temperature,
      max_new_tokens=max_new_tokens,
  )[0]

  print(tokenizer.decode(predicted_token_ids))
```

In [ ]:
#@title Tokenize text input
print(f'prompt = {phi3_prompt}')
print()

print('Tokenizing the prompt......')
phi3_tokens = phi3_tokenizer(phi3_prompt, return_tensors="pt")
print(phi3_tokens)

prompt = <|user|>
Write a movie review. The movie should be an actual movie. The review should start with the word "I".<|end|>
<|assistant|>


Tokenizing the prompt......
{'input_ids': tensor([[32010, 14350,   263, 14064,  9076, 29889,   450, 14064,   881,   367,
           385,  3935, 14064, 29889,   450,  9076,   881,  1369,   411,   278,
          1734,   376, 29902,  1642, 32007, 32001]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1]])}


In [ ]:
#@title The LLM generates token ids
phi3_tokens = {k: v.to(device) for k, v in phi3_tokens.items()}
phi3_model = phi3_model.to(device)

predicted_phi3_token_ids = phi3_model.generate(
      **phi3_tokens,
      do_sample=True,
      #Temperature closer to 1 implies less deterministic output
      temperature=0.9,
      max_new_tokens=16,
  )
predicted_phi3_token_ids

tensor([[32010, 14350,   263, 14064,  9076, 29889,   450, 14064,   881,   367,
           385,  3935, 14064, 29889,   450,  9076,   881,  1369,   411,   278,
          1734,   376, 29902,  1642, 32007, 32001,   306, 10325, 20654,   278,
          2706,   376,  1576, 28548,   845,   804,  4367,   331,   683,  1699,
         10624,   491]], device='cuda:0')

In [ ]:
#@title Translate generated token ids back to text
phi3_tokenizer.decode(predicted_phi3_token_ids[0])

'<|user|> Write a movie review. The movie should be an actual movie. The review should start with the word "I".<|end|><|assistant|> I recently watched the film "The Shawshank Redemption," directed by'

In [ ]:
#@title Controling text generation randomness with temperature
#Coherent Output
#Less Random
#Determinstic
for _ in range(3):
  print('-------------------------------------------------------------------')
  print(f'temperature = 1e-9')
  llm_generate_text(phi3_model, phi3_tokenizer, phi3_prompt, temperature=1e-9, max_new_tokens=16)
print('###################################################################')

#Less Coherent Output
#More Random
#Less Determinstic
for _ in range(3):
  print('-------------------------------------------------------------------')
  print(f'temperature = 1.0')
  llm_generate_text(phi3_model, phi3_tokenizer, phi3_prompt, temperature=1.0, max_new_tokens=16)
print('###################################################################')

#Non Coherent Output
#Extremely Random
for _ in range(3):
  print('-------------------------------------------------------------------')
  print(f'temperature = 10.0')
  llm_generate_text(phi3_model, phi3_tokenizer, phi3_prompt, temperature=10.0, max_new_tokens=16)
print('###################################################################')

-------------------------------------------------------------------
temperature = 1e-9
<|user|> Write a movie review. The movie should be an actual movie. The review should start with the word "I".<|end|><|assistant|> I recently had the pleasure of watching "The Shawshank Redemption,"
-------------------------------------------------------------------
temperature = 1e-9
<|user|> Write a movie review. The movie should be an actual movie. The review should start with the word "I".<|end|><|assistant|> I recently watched the movie "The Shawshank Redemption," directed by
-------------------------------------------------------------------
temperature = 1e-9
<|user|> Write a movie review. The movie should be an actual movie. The review should start with the word "I".<|end|><|assistant|> I recently had the pleasure of watching the film "The Shawshank Redem
###################################################################
-------------------------------------------------------------------
tem

## Inferenece
- `temperature:1e-9`(Determinsitic Output for every iteration)
- `temperature:1`(Less Deterministic Output for every iteration)
- `temperature:10`(Random Ouput for every iteration)